In [28]:
import numpy as np
import pandas as pd
import matplotlib
import plotly
import os
import glob
import plotly.express as px

In [29]:
#path to the experiments log folder
log_data_dir = "/nfs/lslab2/fireline/simharness/experiments/"

#model type in experiment folder
model = 'DQN'

#output folder
outputs = "exp_reports/"

#enter the experiment number here
exp = '2023-09-27_19-44-13'

In [30]:
#concatenate the experiment directory path
exp_dir = log_data_dir + exp + "/" + model + "/"

#get the name of the directory that holds the gifs and csv logs
log_dir_name = [s for s in os.listdir(exp_dir) if model in s][0]

#get the full path of the log directory 
log_dir = exp_dir + log_dir_name + "/"

print("Directory with callback logs and gifs:")
print(log_dir)
print("Directory Contains:")
print(os.listdir(log_dir))



Directory with callback logs and gifs:
/nfs/lslab2/fireline/simharness/experiments/2023-09-27_19-44-13/DQN/DQN_simharness2.environments.reactive.ReactiveHarness_48aba_00000_0_2023-09-27_19-44-37/
Directory Contains:
['params.json', 'params.pkl', 'result.json', 'events.out.tfevents.1695843984.sh2DG', 'progress.csv', 'gifs', 'sim_data', 'benchsim_data', 'checkpoint_000020', 'checkpoint_000040', 'checkpoint_000060']


### Visualize the Benchsim Results

In [31]:
#get the most recent log in the benchsim folder for the bench_sim stats
##this assumes that all the benchsim stats are the same
benchsim_eval_log = max(([os.path.join((log_dir + "benchsim_data/"), basename) for basename in os.listdir((log_dir + "benchsim_data/"))]), key=os.path.getctime)
print(benchsim_eval_log)

#turn the benchsim data into a dataframe
benchsim_data = pd.read_csv(benchsim_eval_log, header=0, index_col=0)
print(benchsim_data.head())

/nfs/lslab2/fireline/simharness/experiments/2023-09-27_19-44-13/DQN/DQN_simharness2.environments.reactive.ReactiveHarness_48aba_00000_0_2023-09-27_19-44-37/benchsim_data/eval_iter_7.csv
          timestep  burned  burning  unburned
sim_step                                     
0                0       0        3     16381
1                4       0        6     16378
2                8       0       10     16374
3               12       0       15     16369
4               16       1       20     16363


In [32]:
benchsim_plot = px.line(benchsim_data, x="timestep", y=['unburned', 'burning', 'burned'], title='Benchsim Analytics')
benchsim_plot.show()

### Visualize the Simharness Agent Results

#### Most Recent Agent Results

In [33]:
#get the most recent log in the sim folder for the agent_sim stats
#agent_eval_log = max(([os.path.join((log_dir + "sim_data/"), basename) for basename in os.listdir((log_dir + "sim_data/"))]), key=os.path.getctime)
#print(agent_eval_log)

#or choose an agent_eval_log
##print possible files
#print(((log_dir + "sim_data/")))
agent_iter_number = 7
agent_eval_log = log_dir + "sim_data/eval_iter_" + str(agent_iter_number) + ".csv"

#turn the benchsim data into a dataframe
agent_data = pd.read_csv(agent_eval_log, header=0, index_col=0)

In [34]:
#merge in the benchmark sim data to create the simharness_analytics dataframe
benchmark_analytics = benchsim_data[['timestep', 'burned', 'burning', 'unburned']].copy()
benchmark_analytics.rename(columns={"burned": "unmitigated_burned", "burning": "unmitigated_burning", "unburned": "unmitigated_unburned"}, inplace=True)
eval_analytics = agent_data.merge(benchmark_analytics, how='inner', on='timestep')
print(eval_analytics.head())

   timestep  burned  burning  unburned  mitigated  agent_interactions  \
0         0       0        3     16381          0                   0   
1         4       0        6     16376          2                   2   
2         8       0       10     16370          4                   3   
3        12       0       15     16364          5                   2   
4        16       1       20     16357          6                   2   

   agent_movements  unmitigated_burned  unmitigated_burning  \
0                1                   0                    3   
1                4                   0                    6   
2                3                   0                   10   
3                3                   0                   15   
4                4                   1                   20   

   unmitigated_unburned  
0                 16381  
1                 16378  
2                 16374  
3                 16369  
4                 16363  


In [35]:
sh_burned_plot = px.line(eval_analytics, x="timestep", y=['mitigated','burned','unmitigated_burned'], title='SimHarness Performance Analysis')
sh_burned_plot.show()

In [36]:
sh_agent_plot = px.line(eval_analytics, x="timestep", y=['mitigated','agent_interactions','agent_movements'], title='SimHarness Performance Analysis')
sh_agent_plot.show()

## Cumulative Agent Analysis

In [37]:
agent_log_dir = log_dir + "sim_data/"
simharness_analytics = pd.DataFrame(columns=['Eval_Num', 'Num_timesteps', 'Num_burned', 'Num_unburned', 'Num_mitigated'])

for basepath in os.listdir(agent_log_dir):
    #get the eval_num from the csv file name
    eval_num = basepath.replace('eval_iter_', '')
    eval_num = eval_num.replace('.csv', '')
    eval_num = int(eval_num)
    
    #take the last row of the eval_iter_#.csv and translate those values into our dataframe
    single_agent_data = pd.read_csv((agent_log_dir + basepath), header=0, index_col=0)
    last_row = single_agent_data.iloc[-1:]
    #print(type(last_row.timestep.iloc[0]))
    final_run_data= [eval_num, int((last_row.timestep.iloc[0])), int((last_row.burned.iloc[0])), int((last_row.unburned.iloc[0])),int((last_row.mitigated.iloc[0]))]
    
    #place that last row data into the dataframe
    simharness_analytics.loc[len(simharness_analytics)] = final_run_data

print(simharness_analytics.head())
    

   Eval_Num  Num_timesteps  Num_burned  Num_unburned  Num_mitigated
0         1           1032       16306             0             78
1         2           1032       16255             0            129
2         3           1032       16230             0            154
3         4           1032       16322             0             62
4         5           1032       16214             0            170


In [38]:
sh_cumulative_plot = px.line(simharness_analytics, x="Eval_Num", y=['Num_burned','Num_unburned','Num_mitigated'], title='SimHarness Performance Analysis')
sh_cumulative_plot.show()